[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/06-agentic-ai/01-why_agents_need_deterministic_grounding.ipynb)

In [1]:
# !pip install mbox openai python-dotenv

# Why Agents Need Deterministic Grounding

An agent built on an LLM is good at understanding what a user wants, and unreliable at holding onto exact facts: a product ID, a price, whether something is even in stock. This notebook is the on-ramp for the rest of `06-agentic-ai/`: it establishes, with real measurements rather than assertions, why every subsequent notebook in this folder routes facts through M|BOX instead of trusting the model's own recall.

In this notebook you will:

1. Try the obvious first instinct: paste your data straight into the prompt
2. Measure what that instinct actually costs, in tokens, latency, and scale
3. See what a prose answer gives you versus what a scored, structured match gives you
4. Get a concrete rule for when to reach for M|BOX instead of the prompt

> Note: this notebook makes real calls to the OpenAI API. To run it, put an `OPENAI_API_KEY` in a `.env` file in this directory.

In [2]:
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

## 1. A catalog that looks like real data

Toy examples with three or four products make any approach look good. `datasets/large_catalog.csv` is 80 rows and deliberately realistic in one specific way: several products come in near-duplicate families, `Extended Battery Pack`, `Extended Battery Pack Pro`, `Extended Battery Pack Mini`, `Extended Battery Pack Heavy Duty`, and so on. Telling these apart correctly, under a typo, is the actual job.

In [3]:
catalog = pd.read_csv("datasets/large_catalog.csv")
print(f"{len(catalog)} rows, {catalog['product_name'].str.split().str[:3].str.join(' ').nunique()} product families")
catalog[catalog["product_name"].str.startswith("Extended Battery Pack")]

80 rows, 10 product families


,product_id,product_name,category,unit_price
0,BAT-101,Extended Battery Pack,Batteries,25.27
1,BAT-102,Extended Battery Pack Pro,Batteries,34.04
2,BAT-103,Extended Battery Pack Mini,Batteries,19.54
3,BAT-104,Extended Battery Pack XL,Batteries,39.44
4,BAT-105,Extended Battery Pack 2-Pack,Batteries,45.45
5,BAT-106,Extended Battery Pack Replacement,Batteries,22.34
6,BAT-107,Extended Battery Pack Compact,Batteries,21.77
7,BAT-108,Extended Battery Pack Heavy Duty,Batteries,36.16


## 2. The obvious first instinct: paste it into the prompt

Every catalog is small until it isn't, and it is tempting to just hand the model the whole thing as text and let it reason in prose. Below, a customer asks about a mistyped, unqualified product name, `"extendd batary pak hevy duty"`, with a budget of \$30, and the model has to pick the right one out of eight very similar names and then check the price itself.

There is no trick question here, we are giving the model everything it needs.

In [4]:
from openai import OpenAI
client = OpenAI()

catalog_text = "\n".join(
    f"- {row.product_name} (id {row.product_id}): ${row.unit_price}" for row in catalog.itertuples()
)

query = "extendd batary pak hevy duty"
prompt = (
    f"Our product catalog:\n{catalog_text}\n\n"
    f"A customer asks: 'Do you have the {query} for under $30?' "
    "Answer with the exact product name, id, and price, and say clearly whether it fits their budget."
)

import time
t0 = time.time()
response = client.chat.completions.create(model="gpt-4o", messages=[{"role": "user", "content": prompt}])
llm_latency = time.time() - t0
llm_prompt_tokens = response.usage.prompt_tokens

print(response.choices[0].message.content)
print(f"\n[{llm_latency:.2f}s, {llm_prompt_tokens} prompt tokens]")

The product you are referring to is the "Extended Battery Pack Heavy Duty" with the ID "BAT-108" and it costs $36.16. This product does not fit the customer's budget, as they are looking for something under $30.

[3.95s, 1393 prompt tokens]


That answer is correct: `Extended Battery Pack Heavy Duty`, and it does not fit a \$30 budget. GPT-4o handled the typo and picked the right family member out of eight lookalikes without help. So if it worked, what's the problem?

The problem is not that this answer is wrong. It's everything the print statement above doesn't show you.

## 3. What that answer actually cost

Look at the token count printed above. That is not a one-time cost, it is the price of *this specific lookup*, paid again on every single turn where the model needs to consult the catalog, because none of it is retained between calls. Below we turn the measured cost-per-row into what it would look like on catalogs you'd actually run a business on.

In [5]:
tokens_per_row = llm_prompt_tokens / len(catalog)
print(f"Measured: {llm_prompt_tokens} tokens for {len(catalog)} rows -> ~{tokens_per_row:.1f} tokens/row\n")

for n_rows in [80, 1_000, 5_000, 50_000]:
    projected = int(tokens_per_row * n_rows)
    flag = "  <- fits in most context windows" if projected < 128_000 else "  <- will not fit in a 128k context window"
    print(f"{n_rows:>7,} rows  ~{projected:>10,} prompt tokens{flag}")

Measured: 1393 tokens for 80 rows -> ~17.4 tokens/row

     80 rows  ~     1,393 prompt tokens  <- fits in most context windows
  1,000 rows  ~    17,412 prompt tokens  <- fits in most context windows
  5,000 rows  ~    87,062 prompt tokens  <- fits in most context windows
 50,000 rows  ~   870,625 prompt tokens  <- will not fit in a 128k context window


That projection is linear extrapolation from a real measurement, not a guess, and it is already optimistic: it assumes the catalog text stays exactly this compact as it grows, and it only counts the catalog itself, not the system prompt, the conversation history, or the model's own reasoning. A real product or customer table is routinely in the tens or hundreds of thousands of rows. This approach runs out of road long before then, and every row you do manage to fit is billed again on the next turn, and the one after that.

## 4. What that answer gives your code

Set the token cost aside. Even at 80 rows, where the model answered correctly, look again at what it actually handed back: a paragraph. To let an agent act on this, some downstream code has to decide, from that paragraph, whether to proceed, ask for clarification, or escalate. That means either trusting the model's own framing of its confidence, which is well known to be poorly calibrated, or regex-ing a sentence that has no guaranteed shape.

Here is the same lookup through M|BOX.

In [6]:
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

t0 = time.time()
index = TableIndexer.create_index(catalog, index_columns=["product_name", "unit_price"], tmp_dir="tmp_index")
build_time = time.time() - t0

fields = [
    TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                            minimum_quality=0, weight=70, mode=TableRecallMode.APPROX),
    TableRecallFieldConfig(input_column="unit_price", indexed_column="unit_price",
                            minimum_quality=0, weight=30, mode=TableRecallMode.NUM_LOWER),
]
config = TableRecallConfig(fields=fields, max_results=3, min_total_match_value=0, include_field_scores=True)

t1 = time.time()
constrained = index.match(queries=pd.DataFrame({"product_name": [query], "unit_price": [30]}), config=config)
match_latency = time.time() - t1

print(f"[index build: {build_time*1000:.1f}ms one-time, match: {match_latency*1000:.2f}ms]")
constrained

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object
[index build: 39.6ms one-time, match: 2.57ms]


,query_row,index_row,product_name_candidate,unit_price_candidate,overall_score,product_name_score,unit_price_score
0,0,-1,,,0,0,0


`index_row` is `-1`: nothing satisfies *both* the fuzzy name and the price constraint, because the real Heavy Duty variant is over budget. That is the correct answer, delivered as data your code can check with `if result["index_row"] == -1`, not as a sentence you have to parse.

To find out *why* it's empty, the way the OpenAI-function notebook in this folder does, drop the price constraint and match on name alone.

In [7]:
name_only = TableRecallConfig(
    fields=[TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                    minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
    max_results=1, min_total_match_value=0, include_field_scores=True
)
closest = index.match(queries=pd.DataFrame({"product_name": [query]}), config=name_only)
closest[["product_name_candidate", "product_name_score", "overall_score"]]

,product_name_candidate,product_name_score,overall_score
0,Extended Battery Pack Heavy Duty,53,53


Same real-world fact as the model's paragraph, that a Heavy Duty pack exists and costs more than \$30, but now `product_name_score` is a number an agent's code can threshold on, log, and act on identically every time it runs, not a phrase the next conversation might word differently.

## 5. The comparison, side by side

| | Prompt-stuffed LLM call | M\|BOX match |
|---|---|---|
| Latency for this lookup | measured above, seconds | measured above, milliseconds |
| Cost per lookup | grows with catalog size, paid every turn | flat; index build is a one-time cost, reused indefinitely |
| Ceiling | breaks down once the catalog stops fitting in context | scales to the size of your actual table |
| What you get back | a paragraph | a scored, typed result your code can branch on |
| Reproducibility | can vary in wording between runs | same input, same score, every time |

Neither tool is "better" in the abstract. The model is what you want reasoning about intent, tone, and what the user is actually asking for. M|BOX is what you want holding the facts still underneath it, so the model has something reliable to reason *from*.

## 6. The rule of thumb

Reach for M|BOX, instead of dumping data into the prompt, whenever an agent needs to:

- **Resolve** something the user typed, typos and all, to a specific record in a real table
- **Filter or rank** by a value, like a price or a date, at the same time as matching fuzzy text
- **Act on a confidence score programmatically**, rather than have the model narrate its own certainty
- **Do this over and over**, inside a single conversation or across many, without paying a growing token bill each time

The rest of this folder builds on exactly this foundation: `02` turns an M|BOX search into an OpenAI function the model can call instead of guessing, `03` and `04` expose the same kind of search as a standard MCP tool, first a single one, then several views over one index built just once, `05` applies this same grounding principle to retrieval-augmented generation, `06` uses the score itself to decide when an agent should act, ask, or hand off to a human, `07` uses it to catch an LLM's own extraction mistakes before they become actions, and `08` scales it up to resolving entire tables of records at once.